# Advanced PySpark ETL Demo (Databricks-ready)

**Level:** Specialist / Senior Analytics Engineer / Jarosław Błaziński

In [1]:
import logging

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, sum, count, desc,
    dense_rank, when
)
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, DoubleType, StringType
)
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Advanced_PySpark_ETL").getOrCreate()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ETL_LOGGER")


ModuleNotFoundError: No module named 'pyspark'

## Generate example data

In [ ]:
INPUT_PATH = "input_data.csv"   # 500 000 rows, columns: id, amount, category, country

SCHEMA = StructType([
    StructField("id",       IntegerType(), nullable=False),
    StructField("amount",   DoubleType(),  nullable=True),
    StructField("category", StringType(),  nullable=True),
    StructField("country",  StringType(),  nullable=True),
])

df_raw = spark.read.csv(INPUT_PATH, header=True, schema=SCHEMA)
logger.info("Loaded CSV from %s — %d rows", INPUT_PATH, df_raw.count())
df_raw.show(10, truncate=False)


## Parameters & reference data

In [ ]:
VAT_RATE = 0.23

# Countries present in the dataset: DE, DK, SE, PL
country_dim = spark.createDataFrame(
    [
        ("DE", "Germany"),
        ("DK", "Denmark"),
        ("SE", "Sweden"),
        ("PL", "Poland"),
    ],
    ["country", "country_name"]
)


## Data quality metrics

In [ ]:

metrics_before = df_raw.agg(
    count("*").alias("row_count"),
    sum(col("amount").isNull().cast("int")).alias("null_amounts"),
    sum((col("amount") < 0).cast("int")).alias("negative_amounts")
)
metrics_before.show()


## Cleaning & enrichment

In [ ]:

clean_df = df_raw.filter(col("amount").isNotNull()).filter(col("amount") > 0)

enriched_df = (
    clean_df
    .join(country_dim, "country", "left")
    .withColumn("amount_vat", col("amount") * (1 + lit(VAT_RATE)))
)
enriched_df.show()


## Aggregations & window functions

In [ ]:

agg_df = (
    enriched_df
    .groupBy("country", "category")
    .agg(sum("amount").alias("total_amount"))
)

window_spec = Window.partitionBy("country").orderBy(desc("total_amount"))

ranked_df = agg_df.withColumn("rank_in_country", dense_rank().over(window_spec))
ranked_df.show()
